# Session 4: Alternative models, your own analysis, and an intro to machine learning

**TOPIC:** Count models, Logistic regressions, Ordered logistic regressions, *[Break]*, Your own mini-analysis, Decision trees, Random forests, Regression trees, Classification trees.

### Count models

**INTUITION:** The coefficients (β) in a Poisson regression model represent the log change in the expected count of the response variable for a one-unit increase in the predictor variable, holding all other variables constant. The exponentiated intercept represents the expected count when all predictor variables are zero.

In [ ]:
from pathlib import Path
import calendar

import pandas as pd

MONTHS = list(calendar.month_name)[1:]   # ["January", "February", ..., "December"]


def find_data_file(name):
    """Look for data/<name> in this folder, in the folders above it, and one level below."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / "data" / name
        if candidate.exists():
            return candidate
    for candidate in sorted(here.glob(f"*/data/{name}")):
        return candidate
    raise FileNotFoundError(
        f"Could not find data/{name}. Python is currently looking from: {here}\n"
        "Fix: put the 'data' folder next to this notebook (or in a folder above it)."
    )

def load_seatbelts():
    """The monthly data frame we build in Session 1 -- rebuilt on the spot if the file is missing."""
    try:
        df = pd.read_csv(find_data_file("seatbelts_monthly.csv"), parse_dates=["date"])
    except FileNotFoundError:
        try:
            df = pd.read_csv(find_data_file("Seatbelts.csv"))
        except FileNotFoundError:
            import statsmodels.api as sm   # last resort: download the same data (needs internet)
            try:
                df = sm.datasets.get_rdataset("Seatbelts", "datasets").data
            except Exception as err:
                raise FileNotFoundError(
                    "Could not find data/Seatbelts.csv and could not download it either. "
                    "Keep the 'data' folder next to the notebooks (or connect to the internet)."
                ) from err
        df["date"] = pd.date_range("1969-01-01", periods=len(df), freq="MS")
        df["month"] = df["date"].dt.month_name()
    df["month"] = pd.Categorical(df["month"], categories=MONTHS, ordered=True)
    return df


# Load the data saved in Session 1 (load_seatbelts() rebuilds it if the file can't be found).
# Keep this cell: if you jump straight to the mini-analysis further down, run this cell first.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (8, 4.5)

seatbelts_monthly = load_seatbelts()
seatbelts_window = seatbelts_monthly[seatbelts_monthly["date"] >= "1980-01-01"].copy()   # same window as Session 2
print(seatbelts_window.shape)

In [ ]:
# Fit a GLM model: Poisson regression for count data
# VanKilled is a genuine count -- always a non-negative whole number -- which makes it a
# clean fit for Poisson
poisson_model = smf.glm("VanKilled ~ kms + law", data=seatbelts_window, family=sm.families.Poisson()).fit()
print(poisson_model.summary())

# how do you interpret the output of this model?
# (hint: np.exp(poisson_model.params) turns the log-scale coefficients into multiplicative effects on the expected count)
print(np.exp(poisson_model.params).round(3))

### Logistic regressions

**INTUITION:** Logistic regression is used when the response (dependent) variable is binary, i.e. has two possible outcomes (we use ordered logistic regression when we have more categories).

In [ ]:
# Create a binary response variable
# We will create a binary variable high_deaths which is 1 if DriversKilled is above the median value and 0 otherwise.
seatbelts_window["high_deaths"] = (seatbelts_window["DriversKilled"] > seatbelts_window["DriversKilled"].median()).astype(int)

# Fit a logistic regression model
# We will use kms (distance driven) and law as predictor variables.
logistic_model = smf.logit("high_deaths ~ kms + law", data=seatbelts_window).fit()
print(logistic_model.summary())

# Interpretation:
# Intercept: The log-odds of a month being an above-median-deaths month when kms and law are zero.
# Coefficients: The change in log-odds for a one-unit increase in the predictor variable.

### Ordered logits

**INTUITION:** Ordered logistic regression (also known as ordinal logistic regression) is used when the dependent variable is ordinal, meaning it has ordered categories.

In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Create an ordinal response variable based on DriversKilled
# We'll categorize it into three ordinal categories: low, medium, and high (roughly equal-sized groups).
seatbelts_window["deaths_cat"] = pd.qcut(seatbelts_window["DriversKilled"], [0, 1/3, 2/3, 1], labels=["Low", "Medium", "High"])
seatbelts_window["deaths_cat"] = pd.Categorical(seatbelts_window["deaths_cat"], categories=["Low", "Medium", "High"], ordered=True)
print(seatbelts_window["deaths_cat"].value_counts().sort_index())

# Fit an ordered logistic regression model
# We'll use kms (rescaled to 1000s -- optimizers can struggle to converge when
# predictors are on very different scales) and law as predictor variables.
seatbelts_window["kms_k"] = seatbelts_window["kms"] / 1000
ord_logit_model = OrderedModel.from_formula("deaths_cat ~ kms_k + law", data=seatbelts_window, distr="logit").fit(method="bfgs", disp=False)
print(ord_logit_model.summary())
# Coefficients: Interpret the direction and significance of the predictors.
# Thresholds: These are the points on the latent variable that separate the categories.
# statsmodels reports the first threshold, then the LOG of the gap to the next one. To read the
# actual cut-points (what R's clm() shows directly):
print(ord_logit_model.model.transform_threshold_params(ord_logit_model.params).round(3))

### Task: extend the ordered logit model

`ord_logit_model` predicts `deaths_cat` (Low/Medium/High drivers killed) from `kms_k` (distance driven, 1000s) and `law`. `seatbelts_window` also has `PetrolPrice`.

1. Fit a new model called `ord_logit_model2` that adds `PetrolPrice` as a third predictor, using `OrderedModel.from_formula()` just like above.
2. Print its `summary()`. Find the row for `PetrolPrice`: is its coefficient positive or negative? Is it statistically significant (check the `P>|z|` column)?
3. In one sentence: does `law`'s coefficient (and significance) survive adding `PetrolPrice` to the model?

In [ ]:
# your code here
# ord_logit_model2 = OrderedModel.from_formula("... ~ ... + ... + ...", data=seatbelts_window, distr="logit").fit(method="bfgs", disp=False)
# print(ord_logit_model2.summary())

### Task: full mini-analysis — a preamble to the policy impact

Every session this week has circled back to one real question: did the 1983 seatbelt law reduce road deaths? This task is your chance to run a complete analysis of *your own* question about this data — from a theory, through a test, to an honest interpretation. It is a first look, not a rigorous causal evaluation.

**Ground rules — this is an exploratory project.** You have observational monthly data and linear regression. That is enough to describe patterns and to check whether they hold up, but not to prove that anything *caused* anything. So:

- Write "is associated with", not "reduces" or "causes", unless you can argue the alternatives away.
- For every result, name **at least two alternative explanations** (think: season, the long-run downward trend in deaths, other things that changed at the same time as the law, the arrow running the other way).
- Consecutive months are not independent of each other (this is a time series), so p-values here are optimistic. Treat them as rough guides, not verdicts.

**Pick a starting question** (or invent your own from the variables in `seatbelts_monthly`):

| # | Question | Outcome ~ predictor |
|---|---|---|
| 1 | Did the law reduce drivers killed? | `DriversKilled ~ law` |
| 2 | Was the change concentrated among front-seat casualties rather than rear-seat ones? | `front ~ law`, compared with `rear ~ law` |
| 3 | Do higher petrol prices reduce how much people drive? | `kms ~ PetrolPrice` |
| 4 | Does more driving mean more deaths? | `DriversKilled ~ kms` |
| 5 | Did the death *rate per distance driven* fall after the law? | `rate ~ law` (`rate` is built for you below) |
| 6 | Which months are deadliest, and by how much? | `DriversKilled ~ month` |
| 7 | Were deaths already falling *before* the law? | `DriversKilled ~ time`, using only the months where `law == 0` |
| 8 | Did van-driver deaths move around the law too? (Look up who the law covered before you interpret the answer.) | `VanKilled ~ law` |
| 9 | Before the law: what predicts deaths, and is anything better than `kms` or `PetrolPrice`? (Try `DriversKilled`, `VanKilled`, or another outcome. Careful: the outcome columns count overlapping accidents, so don't predict one with another.) | outcome `~` any combination of `kms`, `PetrolPrice`, `month`, `time`; only months where `law == 0` |
| 10 | Your own question | |

**Steps**

1. **Theory.** In 2-3 sentences, say *why* you expect a relationship. Give a mechanism, not just a guess.
2. **Hypothesis.** One specific, testable statement (H1: ...). Say which variable is the outcome and which is the predictor.
3. **Feasibility check.** Explore the variables (`.info()`, `.describe()`, a quick plot). Right type? Enough variation? Revise your hypothesis now if it isn't testable as stated.
4. **Descriptive statistics.** Describe your variables before testing anything.
5. **Test.** Fit your model with `smf.ols()`.
6. **Robustness check** (the one habit to take away from this task): refit your model adding `month`, then adding `month` **and** `time`. Does your coefficient keep its sign and rough size? If it changes a lot or flips, your first result was fragile.
7. **Interpret.** What does the result say? Does it support your theory? Which of your alternative explanations does the robustness check rule out, and which does it leave open?

In [ ]:
# --- Setup for this task. (Jumping straight here? Run the first code cell of this notebook first.) ---
seatbelts_monthly = load_seatbelts()

# Three new variables you can use:
seatbelts_monthly["year"] = seatbelts_monthly["date"].dt.year
seatbelts_monthly["time"] = np.arange(1, len(seatbelts_monthly) + 1)   # month counter: 1 = Jan 1969, ..., 192 = Dec 1984
seatbelts_monthly["rate"] = seatbelts_monthly["DriversKilled"] / seatbelts_monthly["kms"] * 1000   # deaths per 1,000 distance units

# The same 1980-onward window we used in Session 2:
seatbelts_window = seatbelts_monthly[seatbelts_monthly["date"] >= "1980-01-01"].copy()

# The months before the law (for questions 7 and 9):
pre_law = seatbelts_monthly[seatbelts_monthly["law"] == 0]
print(seatbelts_monthly.shape, seatbelts_window.shape, pre_law.shape)

In [ ]:
# Tools you can use (edit the variable names for your own question). This example runs as it is:
m1 = smf.ols("DriversKilled ~ law", data=seatbelts_monthly).fit()                   # a simple model
m2 = smf.ols("DriversKilled ~ law + month", data=seatbelts_monthly).fit()           # controlling for season
m3 = smf.ols("DriversKilled ~ law + month + time", data=seatbelts_monthly).fit()    # ...and for the long-run trend
print(m3.summary())

# the three models side by side (month dummies hidden, but flagged)
print(summary_col(
    [m1, m2, m3], stars=True, float_format="%.3f", model_names=["(1)", "(2)", "(3)"],
    regressor_order=["law", "time", "Intercept"], drop_omitted=True,
    info_dict={"Month dummies": lambda m: "Yes" if any("month" in name for name in m.params.index) else "No"},
))

# a quick scatter plot with a fitted line
fig, ax = plt.subplots()
sns.regplot(data=seatbelts_monthly, x="kms", y="DriversKilled", ax=ax)
plt.show()
# Use the full series (seatbelts_monthly) or the 1980-onward window (seatbelts_window): try both,
# and say which you prefer and why.

In [ ]:
# 1. Theory (write as a comment)


# 2. Hypothesis (write as a comment)


# 3. Feasibility check
# seatbelts_monthly.info()


# 4. Descriptive statistics


# 5. Test


# 6. Robustness: refit with + month, then + month + time


# 7. Interpretation (write as a comment)

## Machine learning: intuition, not just code

We may not get through all the code below in class — that's fine. What matters is that you leave with a sense of **when** you'd reach for these tools, **what** they actually do, and **how** they fit alongside everything else you've learned this week, rather than a step-by-step of the syntax.

Machine learning is a subset of AI that builds systems which learn patterns from data rather than being told the rule in advance (*less weight on theory, more weight on the data itself*). Everything you've done so far in this bootcamp starts from a theory: you specify a model (`ols`, `glm`, `OrderedModel`...) because you have a hypothesis about *which* variables matter and *how*. Trees and forests flip that around: you hand them the data and let the algorithm find the splits, interactions, and nonlinearities for you. That's powerful when you don't know the functional form in advance, or when there are too many possible interactions to specify by hand — but it also means you're trading interpretability for flexibility.

**A concrete example, closer to home than iris flowers:**

Say you ran an experiment testing whether an AI chatbot's persuasive messaging shifts people's opinions — exposing people to different AI-generated messages and measuring opinion change (this is, literally, the kind of RCT used to study AI persuasion). A **classification or regression tree** could take respondent features (prior attitude strength, political leaning, education) and message features (tone, length, which model generated it) and show you, in one picture, which *combinations* predict the biggest shift — no need to guess the interactions ahead of time. A **random forest** averages hundreds of such trees for a more reliable prediction, and its "variable importance" score tells you which levers matter most overall.

This is exactly the machinery behind **causal forests** (Athey & Wager's generalized random forests): the same tree-splitting idea, but built specifically to estimate *who* a treatment affects most — heterogeneous treatment effects, the same concept behind the HTE work you may have seen elsewhere, just estimated with trees instead of a handful of pre-specified subgroups. This is increasingly how AI systems and AI policy interventions get evaluated in practice: not just "did the intervention work on average," but "for whom did it work, and why."

**The trade-off to hold onto:** ML is a *prediction* and *pattern-discovery* tool. It doesn't replace the theory-and-design work from the rest of this bootcamp — you still need something like a randomized experiment (or, for observational data like this week's seatbelts data, a genuinely careful design) to trust that a pattern reflects a real effect rather than a coincidence in the training data. Think of it as a complement: theory tells you where to look; trees and forests are good at telling you what you'd have missed by only looking where theory told you to.

### Classification trees

**INTUITION:** use in predicting categorical outcomes. For example, we will see how the tree splits based on `kms` and `law` to classify whether a month is a "High" or "Low" deaths month.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

features = ["kms", "law"]

# Create a binary response variable based on DriversKilled
seatbelts_window["high_deaths_lbl"] = np.where(seatbelts_window["DriversKilled"] > seatbelts_window["DriversKilled"].median(), "High", "Low")

# Fit a classification tree.
# Unlike R's rpart (which stops growing the tree by itself), scikit-learn lets a tree grow until every
# leaf is pure -- which memorizes the data. So we limit it: at most 3 levels deep, at least 7 months per leaf.
class_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=7, random_state=0)
class_tree.fit(seatbelts_window[features], seatbelts_window["high_deaths_lbl"])

fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(class_tree, feature_names=features, class_names=list(class_tree.classes_), filled=True, rounded=True, ax=ax)
plt.show()

### Regression tree

**INTUITION:** use to predict continuous outcomes. For example, the tree splits based on `kms` and `law` to predict the actual number of drivers killed.

In [ ]:
# Fit a regression tree
reg_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=7, random_state=0)
reg_tree.fit(seatbelts_window[features], seatbelts_window["DriversKilled"])

fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(reg_tree, feature_names=features, filled=True, rounded=True, ax=ax)
plt.show()

Decision trees vs random forest:

*Decision trees:*
1. Single model: a decision tree is a single model that splits the data based on feature values to make predictions.
2. Overfitting: decision trees are prone to overfitting, especially if they are deep with many branches. Overfitting means the model performs well on training data but poorly on new, unseen data.

*Random forests:*
1. Ensemble method: a random forest is an ensemble of multiple decision trees. It builds several decision trees and combines their predictions to improve accuracy and robustness.
2. Bootstrapping and aggregation: random forests use bootstrapping (random sampling with replacement) to create multiple subsets of the data. Each tree is trained on a different subset. The final prediction is made by averaging the predictions (regression) or taking a majority vote (classification) from all the trees.
3. Reduced overfitting: by averaging multiple trees, random forests reduce the risk of overfitting and generally provide better generalization performance.

### Random forest *(syntax demo — uses `iris`, not the seatbelts data)*

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load the iris dataset -- a different, unrelated dataset here on purpose: random forests
# are easiest to demonstrate on a case with a clear multi-class outcome and no time
# structure to worry about, which iris gives us for free
iris = load_iris(as_frame=True)
X = iris.data
y = pd.Series(iris.target_names[iris.target], name="Species")

# Split the data into training and test sets (70% training, 30% test); random_state makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

# Fit the random forest model
rf_model = RandomForestClassifier(n_estimators=100, max_features=2, random_state=123)
rf_model.fit(X_train, y_train)
# n_estimators is the number of trees, and max_features is the number of variables randomly sampled as candidates at each split.

# Predict on the test set
predictions = rf_model.predict(X_test)

# Confusion matrix
print(pd.crosstab(y_test.rename("actual"), pd.Series(predictions, index=y_test.index, name="predicted")))

# Accuracy
print("Accuracy:", round(accuracy_score(y_test, predictions), 2))

In [ ]:
# Variable importance plot: how much each variable contributed to the splits
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
importances.plot.barh(ax=ax, color="steelblue")
ax.set(title="Variable importance (random forest)", xlabel="Importance")
plt.tight_layout()
plt.show()